In [1]:
import pandas as pd

df = pd.read_csv("../data/processed/audible_clean.csv")

df_advance = pd.read_csv(
    "../data/processed/audible_advanced_clean.csv"
)

print("Dataset 1 Shape:", df.shape)
print("Dataset 2 Shape:", df_advance.shape)

Dataset 1 Shape: (5439, 5)
Dataset 2 Shape: (4293, 9)


In [2]:
print("Dataset 1 Columns:")
print(df.columns.tolist())

print("\nDataset 2 Columns:")
print(df_advance.columns.tolist())

Dataset 1 Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price']

Dataset 2 Columns:
['Book Name', 'Author', 'Rating', 'Number of Reviews', 'Price', 'Description', 'Listening Time', 'Ranks and Genre', 'Listening_Time_Minutes']


In [3]:
df_advance[[
    "Book Name",
    "Author",
    "Description",
    "Ranks and Genre"
]].head(5)

,Book Name,Author,Description,Ranks and Genre
0,Think Like a Monk: The Secret of How to Harnes...,Jay Shetty,"Over the past three years, Jay Shetty has beco...",",#1 in Audible Audiobooks & Originals (See Top..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Héctor García,Brought to you by Penguin.,",#2 in Audible Audiobooks & Originals (See Top..."
2,The Subtle Art of Not Giving a F*ck: A Counter...,Mark Manson,"In this generation-defining self-help guide, a...",",#3 in Audible Audiobooks & Originals (See Top..."
3,Atomic Habits: An Easy and Proven Way to Build...,James Clear,Brought to you by Penguin.,",#5 in Audible Audiobooks & Originals (See Top..."
4,Life's Amazing Secrets: How to Find Balance an...,Gaur Gopal Das,"Stop going through life, Start growing throug...",",#6 in Audible Audiobooks & Originals (See Top..."


In [4]:
# Display complete genre/rank information for 10 books
for value in df_advance["Ranks and Genre"].head(10):
    print(value)
    print("-" * 80)

,#1 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Personal Success,#1 in Stress Management,#2 in Society & Culture (Books)
--------------------------------------------------------------------------------
,#2 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Meditation (Audible Audiobooks & Originals),#1 in Self-Esteem,#2 in Personal Success
--------------------------------------------------------------------------------
,#3 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#3 in Personal Success,#16 in Personal Development & Self-Help
--------------------------------------------------------------------------------
,#5 in Audible Audiobooks & Originals (See Top 100 in Audible Audiobooks & Originals),#1 in Psychology (Audible Audiobooks & Originals),#3 in Stress Management,#4 in Personal Success
--------------------------------------------------------------------------------
,#

In [5]:
import re

def extract_genres(rank_genre):
    if pd.isna(rank_genre):
        return []

    genres = re.findall(r"#\d+\s+in\s+([^,]+)", rank_genre)

    # Remove the overall Audible ranking
    genres = [
        genre.strip()
        for genre in genres
        if "Audible Audiobooks & Originals" not in genre
    ]

    return genres

df_advance["Genres"] = df_advance["Ranks and Genre"].apply(extract_genres)

df_advance[["Book Name", "Genres"]].head(10)

,Book Name,Genres
0,Think Like a Monk: The Secret of How to Harnes...,"[Personal Success, Stress Management, Society ..."
1,Ikigai: The Japanese Secret to a Long and Happ...,"[Self-Esteem, Personal Success]"
2,The Subtle Art of Not Giving a F*ck: A Counter...,"[Personal Success, Personal Development & Self..."
3,Atomic Habits: An Easy and Proven Way to Build...,"[Stress Management, Personal Success]"
4,Life's Amazing Secrets: How to Find Balance an...,"[Literary Essays, Essays (Books), Spiritualism]"
5,Sapiens,"[History of Civilization, Anthropology (Books)]"
6,Extraordinary Leadership,"[Leadership, Business Careers]"
7,The Intelligent Investor Rev Ed.,"[Personal Finance, Investing & Trading, Analys..."
8,Rich Dad Poor Dad: What the Rich Teach Their K...,"[Personal Finance, Investing & Trading, Person..."
9,The 5AM Club: Own Your Morning. Elevate Your L...,"[Sleep Disorders, Personal Success, Healthy Li..."


In [6]:
# Combine Description and Genres
df_advance["Content"] = (
    df_advance["Description"].fillna("") + " " +
    df_advance["Genres"].apply(lambda x: " ".join(x))
)

# Display the combined content
df_advance[["Book Name", "Content"]].head(5)

,Book Name,Content
0,Think Like a Monk: The Secret of How to Harnes...,"Over the past three years, Jay Shetty has beco..."
1,Ikigai: The Japanese Secret to a Long and Happ...,Brought to you by Penguin. Self-Esteem Persona...
2,The Subtle Art of Not Giving a F*ck: A Counter...,"In this generation-defining self-help guide, a..."
3,Atomic Habits: An Easy and Proven Way to Build...,Brought to you by Penguin. Stress Management P...
4,Life's Amazing Secrets: How to Find Balance an...,"Stop going through life, Start growing throug..."


In [10]:
#TF-IDF Vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
tfidf = TfidfVectorizer(stop_words="english")

# Convert book content into TF-IDF vectors
tfidf_matrix = tfidf.fit_transform(df_advance["Content"])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (4293, 14732)
